# 02a — Lakeflow Connect: Managed SQL Server Connector

**Format:** ~30 min, 👨‍🏫 trainer-driven demo — participants follow along and run the verification cells.

RetailHub's finance team keeps its wholesale channel in an operational **Azure SQL Database**
(AdventureWorksLT sample, CDC enabled). Instead of hand-writing JDBC extract jobs, we ingest it with
**Lakeflow Connect** — the managed SQL Server connector — straight into Unity Catalog bronze tables.

| Section | What happens | Who acts |
|---|---|---|
| A | Decision framework — which ingestion tool when | discussion |
| B | Prerequisites & architecture (connection → gateway → ingestion pipeline) | theory |
| C | UI walkthrough — create the connection & the ingestion pipeline | 👨‍🏫 TRAINER |
| D | Verify the landed tables with SQL | **everyone** |
| E | Live CDC round-trip — UPDATE in the source, change appears in bronze | 👨‍🏫 TRAINER + everyone |
| F | Exam call-out + cost note | wrap-up |

> **Infrastructure:** the source database is provisioned by `infra/terraform`
> (Azure SQL Server + AdventureWorksLT) plus `infra/sql/create_login_master.sql` + `enable_cdc.sql` (CDC + the
> low-privilege `lakeflow_connect` user). The trainer reads **host, database and user**
> from `terraform output lakeflow_connect_connection_summary`; the user's **password**
> is the one set in `create_login_master.sql`.

## A. Decision framework — picking the right ingestion tool

Ingestion is **21% of the exam**. The recurring question shape: *"Data lives in X, lands in Y, with Z constraints — which tool?"*

| Tool | Source type | Effort | Incremental? | When to pick it |
|---|---|---|---|---|
| **Auto Loader** (`cloudFiles`) | **Files in object storage** (S3/ADLS/GCS/Volumes) — JSON, CSV, Parquet, Avro… | Low (code) | Yes — file discovery + checkpoints | Continuous/scheduled file feeds; schema evolution; exactly-once file ingestion |
| **Lakeflow Connect — standard connectors** | Message buses & CDC feeds you wire yourself (e.g. Kafka, Kinesis) via SDP/Structured Streaming | Medium (code) | Yes | Streaming sources where you control the reader in pipeline code |
| **Lakeflow Connect — managed connectors** | **SaaS apps & databases**: Salesforce, Workday, ServiceNow, Google Analytics, **SQL Server / Azure SQL** … | Low (UI/API, no code) | Yes — snapshots + **CDC / change tracking** | Fully managed ingestion of enterprise apps and databases into UC, minimal engineering |
| **Partner tools** (Fivetran, Qlik, Informatica…) | Hundreds of long-tail sources | Low (SaaS) | Yes (tool-dependent) | Source not covered by Lakeflow Connect; existing partner licence/team |
| **JDBC in a notebook** (`spark.read.jdbc`) | Any JDBC database | Medium-high (all yours) | No — full reads unless you build watermarking yourself | One-off exploration, tiny lookup tables; avoid for production feeds |

**Rules of thumb**
- Files landing in storage → **Auto Loader**.
- Enterprise application or operational database, and a managed connector exists → **Lakeflow Connect managed connector**.
- Connector doesn't exist → **partner tool** before hand-rolled JDBC.
- Hand-written JDBC is the last resort — no CDC, no checkpoints, you own retries/schema drift.

### 🤔 Which would you pick? (3 mini-scenarios)

**1.** RetailHub's POS vendor drops ~200 gzipped JSON files per hour into an ADLS container.
You need them in bronze within minutes, with schema evolution handled.

> **Answer:** Auto Loader (streaming or triggered batch mode). Files in object storage are its home turf; notifications/directory listing + checkpoints give exactly-once ingestion.

**2.** The sales team lives in **Salesforce**; leadership wants opportunities and accounts refreshed to the lakehouse every hour, with no engineering budget for custom code.

> **Answer:** Lakeflow Connect **managed Salesforce connector** — fully managed, incremental, UI-configured. JDBC isn't even possible; a partner tool would work but is a second choice when a first-party managed connector exists.

**3.** A legacy on-prem **Informix** database (no Lakeflow Connect connector) must be replicated daily; the company already licenses Fivetran.

> **Answer:** Partner tool (Fivetran). No managed connector exists; hand-written JDBC would mean building incremental logic and monitoring yourself.

## B. Prerequisites & architecture

The managed SQL Server connector is built from **three cooperating objects**:

```
Azure SQL DB (AdventureWorksLT, CDC/change tracking ON)
      │  user: lakeflow_connect (low-privilege: SELECT + metadata/CDC grants)
      ▼
1) UC CONNECTION  (type SQL Server)          ← credentials + host, governed in UC
      ▼
2) INGESTION GATEWAY pipeline                ← classic compute; extracts snapshots
      │                                        + CDC changes from the source
      ▼
   UC VOLUME (staging)                       ← gateway stages raw snapshot/CDC data
      ▼
3) MANAGED INGESTION pipeline (serverless)   ← applies staged changes to targets
      ▼
<catalog>.lakeflow_connect.<tables>          ← governed bronze streaming tables
```

**Prerequisites checklist**
- Unity Catalog and serverless compute enabled in the workspace; permission to create classic clusters (the gateway runs on classic compute).
- Unity Catalog privileges: `CREATE CONNECTION` on the metastore; `USE CATALOG`/`CREATE SCHEMA`/`CREATE TABLE` + `CREATE VOLUME` on the staging & target locations.
- Source database with **CDC** (`sys.sp_cdc_enable_db` / `..._table`) or **change tracking** enabled — after `terraform apply` the trainer runs `infra/sql/create_login_master.sql` (master) + `enable_cdc.sql` (via `sqlcmd`) for the four `SalesLT` tables. Databricks recommends change tracking for tables with a primary key; if both are enabled the connector uses change tracking.
- Network path from the gateway to the SQL endpoint (Terraform opens the firewall for Azure services).

**Why a gateway?** The gateway pipeline runs on dedicated compute *inside your network path* and does the
heavy source-side work (initial snapshot + reading the change feed), decoupled from the serverless
ingestion pipeline that merges those staged changes into Unity Catalog tables. One gateway can serve
the whole class — participants only *read* the landed tables.

## C. 👨‍🏫 TRAINER — UI walkthrough

> Participants: watch and note the objects being created. You will query the results in section D.

### C.1 Create the UC connection

> Field-by-field checklist (what to type, where each value comes from): `infra/LAKEFLOW_CONNECT_CONNECTION.md`

1. **Catalog** → **External Data** → **Connections** → **Create connection**
2. **Connection name:** `retailhub_sqlserver` · **Connection type:** `SQL Server`
3. **Auth:** user/password — host = Terraform output `sql_server_fqdn`
   (e.g. `retailhub-sql-….database.windows.net`), port `1433`, user **`lakeflow_connect`**,
   password = the one set when `infra/sql/create_login_master.sql` created the login
   (host/port/db/user also bundled in the `lakeflow_connect_connection_summary` output)
4. **Test connection** → Create. The connection is now a governed UC object (`SHOW CONNECTIONS`).

### C.2 Create the ingestion pipeline (wizard)
1. **Data Ingestion** (**+ New → Add data**) → **SQL Server** connector
2. Pick connection `retailhub_sqlserver`
3. **Ingestion gateway:** choose the staging **catalog + schema** for the UC Volume the gateway
   will use (e.g. `retailhub_trainer.lakeflow_staging`) — the wizard creates the gateway pipeline
4. **Source:** database = Terraform output `sql_database_name` (AdventureWorksLT) → schema **`SalesLT`** →
   select tables: **`Customer`**, **`Product`**, **`SalesOrderHeader`**, **`SalesOrderDetail`**
5. **Destination:** catalog `retailhub_trainer`, schema **`lakeflow_connect`** *(announced to the class — participants get `SELECT`)*
6. **Schedule:** manual for the demo (production would use a cron trigger) → **Save and run**
7. Watch the two pipelines in **Jobs & Pipelines**: the **gateway** snapshots the 4 tables into the
   staging volume, then the **managed ingestion pipeline** materializes them as streaming tables.

8. **Let the class read the target** (participants query it in section D) — in a SQL cell or the SQL editor,
   using the training group from `00_pre_config` (`TRAINING_GROUP`, e.g. `alt_trn_gr`):
   ```sql
   GRANT USE CATALOG ON CATALOG retailhub_trainer TO `alt_trn_gr`;
   GRANT USE SCHEMA, SELECT ON SCHEMA retailhub_trainer.lakeflow_connect TO `alt_trn_gr`;
   ```

⏱️ First snapshot takes a few minutes — a good moment to recap section A with the class.

## D. Verify the landed tables — run these cells

> ⏳ **Run AFTER the trainer's ingestion pipeline has completed its first update**
> (the trainer will tell you). Everyone has `SELECT` on the shared target schema.

In [0]:
%run ../../setup/00_setup

In [0]:
# Shared target of the trainer's ingestion pipeline (announced by the trainer).
# NOTE: this is the TRAINER's catalog, not your personal one — you only read from it.
LFC_CATALOG = "retailhub_trainer"      # ← adjust if the trainer announces a different catalog
LFC_SCHEMA  = "lakeflow_connect"
LFC         = f"{LFC_CATALOG}.{LFC_SCHEMA}"

display(spark.sql(f"SHOW TABLES IN {LFC}"))

In [0]:
# Row counts of the four ingested SalesLT tables
for t in ["customer", "product", "salesorderheader", "salesorderdetail"]:
    cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {LFC}.{t}").first()["c"]
    print(f"{LFC}.{t:<18} {cnt:>8,} rows")

In [0]:
# A join across the landed tables — top 10 products by wholesale revenue
display(spark.sql(f"""
    SELECT
        p.Name                                  AS product_name,
        COUNT(DISTINCT h.SalesOrderID)          AS orders,
        SUM(d.OrderQty)                         AS units,
        ROUND(SUM(d.LineTotal), 2)              AS revenue
    FROM {LFC}.salesorderdetail d
    JOIN {LFC}.salesorderheader h ON d.SalesOrderID = h.SalesOrderID
    JOIN {LFC}.product          p ON d.ProductID    = p.ProductID
    GROUP BY p.Name
    ORDER BY revenue DESC
    LIMIT 10
"""))

In [0]:
# Inspect table metadata — these are streaming tables maintained by the
# managed ingestion pipeline (see Type / Provider / pipeline properties),
# and DESCRIBE HISTORY shows every pipeline update that touched the table.
display(spark.sql(f"DESCRIBE TABLE EXTENDED {LFC}.customer"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {LFC}.customer LIMIT 5"))

## E. Live CDC round-trip

### 👨‍🏫 TRAINER — run this UPDATE **in Azure SQL** (portal Query editor or `sqlcmd`):

```sql
UPDATE SalesLT.Customer
SET    Phone = '555-TRAINING-42'
WHERE  CustomerID = 1;
```

Then trigger a refresh: **Jobs & Pipelines → ingestion pipeline → Start** (the gateway picks the
change out of the CDC feed; the ingestion pipeline merges it into the target table).

### Everyone — capture *before*, then re-query *after* the refresh:

In [0]:
# BEFORE the trainer's UPDATE + refresh (and again AFTER — compare Phone/ModifiedDate)
display(spark.sql(f"""
    SELECT CustomerID, FirstName, LastName, Phone, ModifiedDate
    FROM {LFC}.customer
    WHERE CustomerID = 1
"""))

In [0]:
# AFTER the pipeline refresh: the CDC change has been merged into bronze.
row = spark.sql(f"SELECT Phone FROM {LFC}.customer WHERE CustomerID = 1").first()
print("Phone now:", row["Phone"])
if row["Phone"] == "555-TRAINING-42":
    print("✅ CDC round-trip observed: source UPDATE → gateway CDC extract → merged into UC bronze")
else:
    print("⏳ Change not visible yet — wait for the pipeline refresh to finish and re-run this cell")

## F. Exam call-out & cost note

### 🎯 Exam (Ingestion — 21%)
- **Prioritization criteria:** files in object storage → Auto Loader; supported SaaS/database →
  **managed connector**; unsupported source → partner tool; hand-rolled JDBC last.
- **Managed connector anatomy:** UC `CONNECTION` (governed credentials) → **ingestion gateway**
  (dedicated pipeline that stages *snapshots + CDC changes* into a **UC Volume**) → **managed
  ingestion pipeline** (merges staged changes into streaming tables).
- Database connectors are **incremental by design** — they consume **CDC / change tracking**,
  not repeated full extracts; targets are **streaming tables**, refreshed by pipeline updates.
- Connections and landed tables are **Unity Catalog objects** → normal `GRANT`s, lineage, audit.

### 💰 Cost note
The demo stack (Azure SQL DB + ingestion gateway compute) runs only for the training:
- SQL DB uses a serverless/auto-pause tier — pennies over 3 days;
- the gateway pipeline uses classic compute and is designed to run continuously (stopping it mid-life can force a full re-snapshot) — **delete the ingestion pipeline and the gateway after the demo** instead of leaving them idle;
- **after Day 3: `terraform destroy`** in `infra/terraform/` (on the instructor checklist).

### 🧯 Fallback (if the connector is unavailable or the source is unreachable)
Trainer shows the pre-recorded run / screenshots, then continues with
[BONUS 03 — Delta Advanced (CDF & CLONE)](BONUS_03_delta_advanced.ipynb).

← [02 — Data Reading & File Formats](02_data_ingestion.ipynb) | **[README](../../../README.md)** | [03 — Delta Lake →](03_delta_lake.ipynb)